In [1]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla

### Function: weighted_jacobi(A, b, x, omega=2/3, num_smooth=3)

The function applies weighted Jacobi iterations to the linear system

$$
Ax=b.
$$

Starting from the current approximation $x$, each smoothing step updates the solution by

$$
x^{k+1} = x^k + \omega D^{-1}(b - Ax^k),
$$

where $D$ is the diagonal part of $A$.

For this assignment, the relaxation parameter is

$$
\omega = \frac{2}{3}.
$$

The purpose of the weighted Jacobi method in the multigrid V-cycle is to smooth high-frequency error components before and after the coarse-grid correction.

#### Input
A : System matrix.

b : Right-hand side vector.

x : Current approximation to the solution.

omega : Relaxation parameter. In this assignment, $\omega = 2/3$.

num_smooth : Number of weighted Jacobi smoothing steps.
#### Output
x : Updated approximation after applying the weighted Jacobi smoothing steps.

In [2]:
def weighted_jacobi(A, b, x, omega=2/3, num_smooth=3):
    D=A.diagonal()
    for _ in range(num_smooth):
        residual=b - A @ x
        x=x + omega * residual / D
    return x

### Function: restriction_operator(n)

The function constructs the full-weighting restriction matrix $R$ used to transfer a residual from a fine grid to a coarse grid.

The fine grid has $n \times n$ interior points, where

$$
n = 2^p - 1.
$$

The corresponding coarse grid has $n_c \times n_c$ interior points, where

$$
n_c = \frac{n-1}{2}.
$$

The restriction matrix maps a fine-grid vector of length $n^2$ to a coarse-grid vector of length $n_c^2$:

$$
R : \mathbb{R}^{n^2} \rightarrow \mathbb{R}^{n_c^2}.
$$

Therefore, the matrix $R$ has size

$$
n_c^2 \times n^2.
$$

Full-weighting restriction means that each coarse-grid value is computed from a weighted average of the corresponding fine-grid point and its neighbouring fine-grid points.

#### Input
n : Number of interior grid points in one spatial direction on the fine grid.

#### Output

R :Restriction matrix of size $n_c^2 \times n^2$, used to transfer vectors from the fine grid to the coarse grid.

In [5]:
def restriction_operator(n):
    nc = (n - 1) // 2
    N_fine = n * n
    N_coarse = nc * nc
    R = sp.lil_matrix((N_coarse, N_fine))
    kc = 0
    for jc in range(nc):
        for ic in range(nc):
            # Corresponding fine-grid point
            i=2*ic + 1
            j=2*jc + 1
            kf=j * n + i
            # Full-weighting 3x3 stencil
            R[kc,kf-n-1] = 1/16
            R[kc,kf-n]=1/8
            R[kc,kf-n+1]= 1/16

            R[kc,kf-1]=1/8
            R[kc,kf]=1/4
            R[kc,kf+1]=1/8

            R[kc,kf+n-1]=1/16
            R[kc,kf+n]=1/8
            R[kc,kf+n+1]= 1/16
            kc += 1

    return R.tocsr()

### Function: build_multigrid_hierarchy(A, n, direct_size=9)

The function build_multigrid_hierarchy builds the sequence of matrices and transfer operators needed for the multigrid V-cycle.

Starting from the finest grid matrix $A$, the function repeatedly constructs coarser grid levels until the system is small enough to be solved directly.

At each level, the coarse-grid matrix is constructed using

$$
A_c = R A_f P,
$$

where $A_f$ is the fine-grid matrix, $R$ is the restriction matrix, and $P$ is the prolongation matrix.

The restriction matrix transfers residuals from the fine grid to the coarse grid, while the prolongation matrix transfers coarse-grid corrections back to the fine grid.

The process continues until the matrix size is less than or equal to direct_size.

#### Input

A : Fine-grid system matrix.

n : Number of interior grid points in one spatial direction on the finest grid.

direct_size : Maximum matrix size for which the system is solved directly.

#### Output

A_list : Matrices at each multigrid level.

R_list : Restriction matrices between consecutive grid levels.

P_list : Prolongation matrices between consecutive grid levels.

In [7]:
def build_multigrid_hierarchy(A, n, direct_size=49):
    A_list=[A]
    R_list=[]
    P_list=[]
    current_A=A
    current_n=n
    while current_A.shape[0] > direct_size and current_n > 3:
        R= restriction_operator(current_n)
        P= 4 * R.T
        coarse_A = R @ current_A @ P
        R_list.append(R)
        P_list.append(P)
        A_list.append(coarse_A.tocsr())
        current_A= coarse_A.tocsr()
        current_n= (current_n - 1) // 2

    return A_list, R_list, P_list

### Function: v_cycle(level, A_list, R_list, P_list, b, x, omega=2/3, pre_smooth=3, post_smooth=3)

The function applies one recursive multigrid V-cycle to approximate the solution of the linear system

$$
Ax=b.
$$

At the current multigrid level, the method first applies weighted Jacobi pre-smoothing. It then computes the residual

$$
r=b-Ax.
$$

The residual is restricted to the next coarser grid. On the coarse grid, the method solves the error equation. If the coarse-grid system is small enough, it is solved directly. Otherwise, the V-cycle is applied recursively.

The coarse-grid error correction is then interpolated back to the finer grid and added to the current approximation. Finally, weighted Jacobi post-smoothing is applied.

#### Input

level : Current multigrid level.

A_list : Matrices at all multigrid levels.

R_list : Restriction matrices between consecutive levels.

P_list : Prolongation matrices between consecutive levels.

b : Right-hand side vector at the current level.

x : Current approximation at the current level.

omega : Weighted Jacobi relaxation parameter.

pre_smooth : Number of weighted Jacobi iterations before the coarse-grid correction.

post_smooth : Number of weighted Jacobi iterations after the coarse-grid correction.

#### Output

x : Updated approximation after one V-cycle.

In [9]:
def v_cycle(level, A_list, R_list, P_list, b, x,omega=2/3, pre_smooth=3, post_smooth=3):
    A = A_list[level]
    # If we are on the coarsest level, solve directly
    if level == len(A_list) - 1:
        x = spla.spsolve(A, b)
        return x
    # 1 Pre-smoothing
    x= weighted_jacobi(A, b, x, omega=omega, num_smooth=pre_smooth)
    # 2Compute residual
    r= b - A @ x
    # 3 Restrict residual to coarse grid
    R= R_list[level]
    r_coarse = R @ r
    # 4 Solve error equation on coarse grid recursively
    e_coarse_0 = np.zeros_like(r_coarse)
    e_coarse = v_cycle(level + 1, A_list, R_list, P_list, r_coarse,e_coarse_0,
        omega=omega, pre_smooth=pre_smooth,post_smooth=post_smooth)
    # 5Interpolate coarse-grid error and correct fine-grid solution
    P= P_list[level]
    x= x + P @ e_coarse
    # 6 Post-smoothing
    x = weighted_jacobi(A, b, x, omega=omega, num_smooth=post_smooth)

    return x